# **Mapping Residential Development Trajectories as Indicators of Emerging and Future Business Markets**

Week 5

•	Hotspot Analysis (Spatial – Clustering Modeling)

# **Loading Data**

In [1]:
import pandas as pd

# Loading the merged dataset
df_merged_final = pd.read_csv('clustered_df_merged_final.csv')

# Displaying the head and info of the loaded DataFrame
print("Head of the loaded merged dataset:")
print(df_merged_final.head())
print("\nInfo of the loaded merged dataset:")
df_merged_final.info()

Head of the loaded merged dataset:
     CSA     CBSA                                               Name  year  \
0  101.0  10180.0  Abilene, TX                                   ...  2024   
1  101.0  10180.0  Abilene, TX                                   ...  2024   
2  101.0  10180.0  Abilene, TX                                   ...  2024   
3  101.0  10180.0  Abilene, TX                                   ...  2024   
4  101.0  10180.0  Abilene, TX                                   ...  2024   

   month        date residential_structure_type  units_count  value_usd State  \
0      1  2024-01-01                     1 Unit         28.0       7967    TX   
1      2  2024-02-01                     1 Unit         31.0       8617    TX   
2      3  2024-03-01                     1 Unit         18.0       5107    TX   
3      4  2024-04-01                     1 Unit         43.0      11923    TX   
4      5  2024-05-01                     1 Unit         33.0       9490    TX   

   units_

## **Data Cleaning**

In [2]:
# Setting Clusters as Object Type
df_merged_final['Cluster'] = df_merged_final['Cluster'].astype('object')

In [4]:
import numpy as np
from sklearn.impute import SimpleImputer
import pandas as pd

# Storing the original column order
original_columns = df_merged_final.columns

# Identifying numeric and non-numeric columns
numeric_cols = df_merged_final.select_dtypes(include=np.number).columns
non_numeric_cols = df_merged_final.select_dtypes(exclude=np.number).columns

# Creating separate DataFrames for numeric and non-numeric data
df_numeric = df_merged_final[numeric_cols].copy()
df_non_numeric = df_merged_final[non_numeric_cols].copy()

# Replacing infinite values with NaN before imputation on numeric columns only
df_numeric.replace([np.inf, -np.inf], np.nan, inplace=True)

# Handling missing values using median imputation on numeric columns only
imputer = SimpleImputer(strategy='median')
df_numeric_imputed_array = imputer.fit_transform(df_numeric)

# Converting the imputed numpy array back to a DataFrame
df_numeric_imputed = pd.DataFrame(df_numeric_imputed_array, columns=numeric_cols, index=df_numeric.index)

# Recombining the imputed numeric DataFrame with the original non-numeric DataFrame
# Using .join to merge on index, then reindex to original column order
df_merged_final = df_numeric_imputed.join(df_non_numeric)

# Ensuring the columns are in their original order
df_merged_final = df_merged_final[original_columns]

# **Hotspot Analysis (Spatial – Clustering Modeling)**

### Calculating Change in Development Intensity

To identify hotspots and cold spots of development intensity,change in development intensity for each CBSA between the earliest and latest years available in the dataset need to be calculated.

In [10]:
# Getting the min and max years from the dataset
min_year = df_merged_final['year'].min()
max_year = df_merged_final['year'].max()

print(f"Earliest year: {min_year}")
print(f"Latest year: {max_year}")

Earliest year: 2020.0
Latest year: 2026.0


In [11]:
# Aggregating development intensity by CBSA and year
development_intensity_agg = df_merged_final.groupby(['CBSA', 'year'])['development_intensity'].mean().reset_index()

# Filtering for min and max years
development_intensity_min_year = development_intensity_agg[development_intensity_agg['year'] == min_year]
development_intensity_max_year = development_intensity_agg[development_intensity_agg['year'] == max_year]

# Merging the two dataframes to calculate the difference
dev_intensity_comparison = pd.merge(
    development_intensity_min_year[['CBSA', 'development_intensity']],
    development_intensity_max_year[['CBSA', 'development_intensity']],
    on='CBSA',
    suffixes=('_min_year', '_max_year')
)

# Calculating the difference in development intensity
dev_intensity_comparison['development_intensity_diff'] = \
    dev_intensity_comparison['development_intensity_max_year'] - \
    dev_intensity_comparison['development_intensity_min_year']

# Displaying the head of the comparison dataframe
print("Head of Development Intensity Comparison:")
display(dev_intensity_comparison.head())

# Checking for any missing values after merging (CBSAs present in one year but not the other)
print("\nInfo of Development Intensity Comparison:")
dev_intensity_comparison.info()

Head of Development Intensity Comparison:


,CBSA,development_intensity_min_year,development_intensity_max_year,development_intensity_diff
0,10180.0,180.893614,215.722494,34.828880
1,10420.0,281.270473,254.140213,-27.130260
2,10500.0,170.944947,190.224996,19.280049
3,10540.0,187.592176,265.232754,77.640578
4,10580.0,184.233129,288.160273,103.927144



Info of Development Intensity Comparison:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 362 entries, 0 to 361
Data columns (total 4 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   CBSA                            362 non-null    float64
 1   development_intensity_min_year  362 non-null    float64
 2   development_intensity_max_year  362 non-null    float64
 3   development_intensity_diff      362 non-null    float64
dtypes: float64(4)
memory usage: 11.4 KB


In [12]:
import geopandas as gpd

# Loading the CBSA shapefile
cbsa_shapefile = gpd.read_file('zip://cb_2020_us_cbsa_500k.zip')

# Displaying the first few rows and information about the GeoDataFrame
print("Head of CBSA Shapefile:")
display(cbsa_shapefile.head())

print("\nInfo of CBSA Shapefile:")
cbsa_shapefile.info()

Head of CBSA Shapefile:


,CSAFP,CBSAFP,AFFGEOID,GEOID,NAME,NAMELSAD,LSAD,ALAND,AWATER,geometry
0,None,43620,310M600US43620,43620,"Sioux Falls, SD","Sioux Falls, SD Metro Area",M1,6670256887,27900457,"POLYGON ((-97.60873 43.81839, -97.60858 43.840..."
1,357,12660,310M600US12660,12660,"Baraboo, WI","Baraboo, WI Micro Area",M2,2153686013,45692969,"POLYGON ((-90.3124 43.64099, -90.3023 43.64116..."
2,184,17460,310M600US17460,17460,"Cleveland-Elyria, OH","Cleveland-Elyria, OH Metro Area",M1,5176805788,5128428315,"POLYGON ((-82.34807 41.42831, -82.344 41.42966..."
3,174,16860,310M600US16860,16860,"Chattanooga, TN-GA","Chattanooga, TN-GA Metro Area",M1,5409949983,126699774,"POLYGON ((-85.87303 35.22068, -85.87303 35.221..."
4,324,29340,310M600US29340,29340,"Lake Charles, LA","Lake Charles, LA Metro Area",M1,6083163521,1766798991,"POLYGON ((-93.92921 29.80295, -93.92799 29.809..."



Info of CBSA Shapefile:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 939 entries, 0 to 938
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   CSAFP     551 non-null    object  
 1   CBSAFP    939 non-null    object  
 2   AFFGEOID  939 non-null    object  
 3   GEOID     939 non-null    object  
 4   NAME      939 non-null    object  
 5   NAMELSAD  939 non-null    object  
 6   LSAD      939 non-null    object  
 7   ALAND     939 non-null    int64   
 8   AWATER    939 non-null    int64   
 9   geometry  939 non-null    geometry
dtypes: geometry(1), int64(2), object(7)
memory usage: 73.5+ KB


In [19]:
# Converting CBSA column in dev_intensity_comparison to string for merging
dev_intensity_comparison['CBSA'] = dev_intensity_comparison['CBSA'].astype(str)
# Removing '.0' suffix from CBSA values to match shapefile format
dev_intensity_comparison['CBSA'] = dev_intensity_comparison['CBSA'].str.replace('\\.0$', '', regex=True)

# Converting CBSAFP in cbsa_shapefile to string for proper matching
cbsa_shapefile['CBSAFP'] = cbsa_shapefile['CBSAFP'].astype(str)

# Merging the development intensity differences with the CBSA shapefile
cbsa_geo_data = pd.merge(
    cbsa_shapefile,
    dev_intensity_comparison,
    left_on='CBSAFP',
    right_on='CBSA',
    how='inner'
)

# Displaying the head and info of the merged GeoDataFrame
print("Head of Merged CBSA GeoData with Development Intensity Difference:")
display(cbsa_geo_data.head())

print("\nInfo of Merged CBSA GeoData:")
cbsa_geo_data.info()

# Checking for any CBSAs that did not merge
print(f"\nNumber of CBSAs in dev_intensity_comparison: {len(dev_intensity_comparison)}")
print(f"Number of CBSAs in cbsa_shapefile: {len(cbsa_shapefile)}")
print(f"Number of CBSAs after merge: {len(cbsa_geo_data)}")

if len(dev_intensity_comparison) != len(cbsa_geo_data):
    print("Warning: Some CBSAs from dev_intensity_comparison did not find a match in the shapefile.")
    missing_cbsas = set(dev_intensity_comparison['CBSA']) - set(cbsa_geo_data['CBSA'])
    if missing_cbsas:
        print(f"Missing CBSAs: {missing_cbsas}")
    else:
        print("All CBSAs from dev_intensity_comparison were merged, but some CBSAs in the shapefile might not have corresponding development intensity data.")

Head of Merged CBSA GeoData with Development Intensity Difference:


,CSAFP,CBSAFP,AFFGEOID,GEOID,NAME,NAMELSAD,LSAD,ALAND,AWATER,geometry,CBSA,development_intensity_min_year,development_intensity_max_year,development_intensity_diff
0,None,43620,310M600US43620,43620,"Sioux Falls, SD","Sioux Falls, SD Metro Area",M1,6670256887,27900457,"POLYGON ((-97.60873 43.81839, -97.60858 43.840...",43620,174.597074,203.094123,28.497049
1,174,16860,310M600US16860,16860,"Chattanooga, TN-GA","Chattanooga, TN-GA Metro Area",M1,5409949983,126699774,"POLYGON ((-85.87303 35.22068, -85.87303 35.221...",16860,131.722649,143.667008,11.944358
2,324,29340,310M600US29340,29340,"Lake Charles, LA","Lake Charles, LA Metro Area",M1,6083163521,1766798991,"POLYGON ((-93.92921 29.80295, -93.92799 29.809...",29340,167.478937,200.757439,33.278502
3,406,25220,310M600US25220,25220,"Hammond, LA","Hammond, LA Metro Area",M1,2049124442,136690438,"POLYGON ((-90.56711 30.8344, -90.56679 30.8602...",25220,197.248200,171.531334,-25.716866
4,122,12060,310M600US12060,12060,"Atlanta-Sandy Springs-Alpharetta, GA","Atlanta-Sandy Springs-Alpharetta, GA Metro Area",M1,22495780629,386874693,"POLYGON ((-85.38647 33.90172, -85.35186 33.901...",12060,170.240827,211.505060,41.264233



Info of Merged CBSA GeoData:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 362 entries, 0 to 361
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   CSAFP                           246 non-null    object  
 1   CBSAFP                          362 non-null    object  
 2   AFFGEOID                        362 non-null    object  
 3   GEOID                           362 non-null    object  
 4   NAME                            362 non-null    object  
 5   NAMELSAD                        362 non-null    object  
 6   LSAD                            362 non-null    object  
 7   ALAND                           362 non-null    int64   
 8   AWATER                          362 non-null    int64   
 9   geometry                        362 non-null    geometry
 10  CBSA                            362 non-null    object  
 11  development_intensity_min_year  362 non-null  

### Hotspot Analysis using Getis-Ord Gi*

The steps are:
1.  **Checking and setting CRS**: Ensures the GeoDataFrame has a projected Coordinate Reference System (CRS) suitable for distance calculations or contiguity.
2.  **Creating Spatial Weights Matrix**: Generating a Queen contiguity weights matrix.
3.  **Performing Getis-Ord Gi***: Applying the `esda.Moran.G_Local` function to calculate the local G-statistic.
4.  **Visualizing Results**: Mapping the hot and cold spots.

In [24]:
import libpysal as lps
import esda
import matplotlib.pyplot as plt

# Checking current CRS and reprojectting if geographic
if cbsa_geo_data.crs.is_geographic:
    print("GeoDataFrame is in a geographic CRS. Reprojecting to a suitable projected CRS (e.g., EPSG:3857).")
    cbsa_geo_data = cbsa_geo_data.to_crs(epsg=3857)
else:
    print(f"GeoDataFrame already has a projected CRS: {cbsa_geo_data.crs}")

# Debugging: Checking if cbsa_geo_data is empty after CRS check
print(f"cbsa_geo_data.empty after CRS check: {cbsa_geo_data.empty}")
print(f"Shape of cbsa_geo_data after CRS check: {cbsa_geo_data.shape}")

if not cbsa_geo_data.empty:
    # Filtering out invalid or empty geometries that might cause issues with weights creation
    initial_count = len(cbsa_geo_data)
    cbsa_geo_data = cbsa_geo_data[cbsa_geo_data.geometry.is_valid & ~cbsa_geo_data.geometry.is_empty]
    if len(cbsa_geo_data) < initial_count:
        print(f"Warning: Removed {initial_count - len(cbsa_geo_data)} invalid or empty geometries.")

    if cbsa_geo_data.empty: # Checking again after cleaning geometries
        print("Cannot perform hotspot analysis: cbsa_geo_data became empty after cleaning invalid geometries.")
    else:
        # 1. Creating a spatial weights matrix (Queen contiguity)
        print("Attempting to create spatial weights matrix...")
        # Reseting index to avoid potential issues with non-unique or non-monotonic indices in weights creation
        cbsa_geo_data = cbsa_geo_data.reset_index(drop=True)
        w = lps.weights.Queen.from_dataframe(cbsa_geo_data)

        # Debugging: Checking if weights matrix is empty
        if w.sparse.nnz == 0:
            print("Warning: The spatial weights matrix is empty. This might happen if no CBSAs share a border, or all are isolated.")
            print("Cannot proceed with Getis-Ord Gi* due to empty weights matrix.")
        else:
            print(f"Spatial weights matrix created with {w.n} observations and {w.sparse.nnz} non-zero connections.")
            # Row-standardize the weights matrix
            w.transform = 'R'

            # 2. Getting the variable for analysis (development_intensity_diff)
            y = cbsa_geo_data['development_intensity_diff'].values

            # 3. Performing Getis-Ord Gi* statistic(G_Local requires the weights matrix and the variable)
            lg = esda.getisord.G_Local(y, w)

            # Adding the Gi* statistics and p-values to the GeoDataFrame
            cbsa_geo_data['g_star_zscore'] = lg.Zs
            cbsa_geo_data['g_star_pvalue'] = lg.p_sim

            # Displaying the head of the GeoDataFrame with the new Gi* columns
            print("\nHead of GeoDataFrame with G* statistics:")
            display(cbsa_geo_data[['NAME', 'development_intensity_diff', 'g_star_zscore', 'g_star_pvalue']].head())

            # Defining significance level (e.g., 90%, 95%, 99%)
            significance_level = 0.05 # For 95% confidence

            # Classifying hot and cold spots
            cbsa_geo_data['hotspot_type'] = 'Not Significant'
            # Checking if z-score and p-value columns are not all NaN before applying conditions
            if not cbsa_geo_data['g_star_zscore'].isnull().all() and not cbsa_geo_data['g_star_pvalue'].isnull().all():
                cbsa_geo_data.loc[(cbsa_geo_data['g_star_zscore'] > 1.96) & (cbsa_geo_data['g_star_pvalue'] < significance_level), 'hotspot_type'] = 'Hot Spot' # 95% confidence, z > 1.96
                cbsa_geo_data.loc[(cbsa_geo_data['g_star_zscore'] < -1.96) & (cbsa_geo_data['g_star_pvalue'] < significance_level), 'hotspot_type'] = 'Cold Spot' # 95% confidence, z < -1.96
            else:
                print("Warning: G_Local results (z-score or p-value) are all NaN, classification skipped.")

            print("\nHotspot Type Distribution:")
            print(cbsa_geo_data['hotspot_type'].value_counts())
else:
    print("Cannot perform hotspot analysis: cbsa_geo_data is empty at the start of the cell.")

GeoDataFrame is in a geographic CRS. Reprojecting to a suitable projected CRS (e.g., EPSG:3857).
cbsa_geo_data.empty after CRS check: False
Shape of cbsa_geo_data after CRS check: (362, 14)
Attempting to create spatial weights matrix...


/tmp/ipykernel_880/4115066832.py:31: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = lps.weights.Queen.from_dataframe(cbsa_geo_data)
/usr/local/lib/python3.13/dist-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 91 disconnected components.
 There are 47 islands with ids: 5, 16, 24, 28, 41, 52, 54, 77, 88, 91, 110, 111, 115, 116, 135, 140, 152, 160, 166, 172, 179, 180, 202, 203, 205, 208, 214, 219, 221, 222, 231, 234, 239, 252, 257, 261, 274, 283, 287, 290, 291, 294, 313, 315, 325, 355, 360.
  W.__init__(self, neighbors, ids=ids, **kw)
/usr/local/lib/python3.13/dist-packages/esda/getisord.py:527: RuntimeWarning: invalid value encountered in divide
  z_scores = (statistic - expected_value) / np.sqrt(expected_variance)


Spatial weights matrix created with 362 observations and 726 non-zero connections.
('WARNING: ', 5, ' is an island (no neighbors)')
('WARNING: ', 16, ' is an island (no neighbors)')
('WARNING: ', 24, ' is an island (no neighbors)')
('WARNING: ', 28, ' is an island (no neighbors)')
('WARNING: ', 41, ' is an island (no neighbors)')
('WARNING: ', 52, ' is an island (no neighbors)')
('WARNING: ', 54, ' is an island (no neighbors)')
('WARNING: ', 77, ' is an island (no neighbors)')
('WARNING: ', 88, ' is an island (no neighbors)')
('WARNING: ', 91, ' is an island (no neighbors)')
('WARNING: ', 110, ' is an island (no neighbors)')
('WARNING: ', 111, ' is an island (no neighbors)')
('WARNING: ', 115, ' is an island (no neighbors)')
('WARNING: ', 116, ' is an island (no neighbors)')
('WARNING: ', 135, ' is an island (no neighbors)')
('WARNING: ', 140, ' is an island (no neighbors)')
('WARNING: ', 152, ' is an island (no neighbors)')
('WARNING: ', 160, ' is an island (no neighbors)')
('WARNING:

/usr/local/lib/python3.13/dist-packages/esda/getisord.py:450: RuntimeWarning: invalid value encountered in divide
  self.z_sim = (self.Gs - self.EG_sim) / self.seG_sim


,NAME,development_intensity_diff,g_star_zscore,g_star_pvalue
0,"Sioux Falls, SD",28.497049,-0.288672,0.405
1,"Chattanooga, TN-GA",11.944358,0.056613,0.444
2,"Lake Charles, LA",33.278502,-0.502269,0.233
3,"Hammond, LA",-25.716866,-2.546459,0.001
4,"Atlanta-Sandy Springs-Alpharetta, GA",41.264233,-0.164813,0.369



Hotspot Type Distribution:
hotspot_type
Not Significant    353
Hot Spot             6
Cold Spot            3
Name: count, dtype: int64


In [32]:
# Saving Hotspot Data
cbsa_geo_data.to_csv("hotspot_cbsa_geo_data.csv", index = False)